# Drug-similarity benchmark for the API → IL recommender

**Question:** does a better way of measuring drug–drug similarity improve counter-ion recommendations?

**Tested:** 7 fingerprints (Morgan2/ECFP4, Morgan3/ECFP6, FeatMorgan/FCFP4, AtomPair, Topological torsion,
MACCS, RDKit path) × 4 similarity measures (Tanimoto, Dice, Cosine, Tversky 0.9/0.1), plus 3D USR and USRCAT —
30 measures × k = 5, 15, 30 neighbours = 90 settings.

**Protocol:** each drug's own known pairs are hidden before it is scored (447 pairs, 186 drugs — same set as the
recommender). To avoid picking a lucky winner, the best setting is chosen on 70% of drugs and scored on the unseen
30%, over 5 random splits.

**Needs:** the 4 data files + `IL_Recommender_v2_pKa.ipynb` (its data-prep code is reused). CPU only, ~5 min.

In [ ]:
!pip install -q rdkit nbformat
import os
NEEDED = ['IL_420_final_pKa_v2.csv', 'API_pKa_table.csv', 'List_B_known_pairs.csv', 'master_IL_ions.csv', 'IL_Recommender_v2_pKa.ipynb']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    from google.colab import files
    print('Upload:', missing)
    for name in files.upload():
        for want in missing:
            stem, ext = os.path.splitext(want)
            if name.startswith(stem) and name.endswith(ext):
                os.replace(name, want)
print('files ready')

## Step 1 — Build all similarity matrices (2D fingerprints, 3D USR/USRCAT)

In [ ]:
import nbformat, numpy as np, pandas as pd, time
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import AllChem, rdMolDescriptors, MACCSkeys, rdFingerprintGenerator as G
RDLogger.DisableLog('rdApp.*')
nb = nbformat.read('IL_Recommender_v2_pKa.ipynb', as_version=4)
code = [c.source for c in nb.cells if c.cell_type == 'code']
g = {}
for src in code[:2]:
    exec('\n'.join(l for l in src.split('\n') if not l.strip().startswith('!')), g)
B, API, POOL, MST = g['B'], g['API'], g['POOL'], g['MST']

APIS = list(B.api_smiles.unique()); idx = {s: i for i, s in enumerate(APIS)}
mols = [Chem.MolFromSmiles(s) for s in APIS]

# ---------- 2D fingerprints ----------
fps = {
 'Morgan2 (ECFP4)':  [G.GetMorganGenerator(radius=2, fpSize=2048).GetFingerprint(m) for m in mols],
 'Morgan3 (ECFP6)':  [G.GetMorganGenerator(radius=3, fpSize=2048).GetFingerprint(m) for m in mols],
 'FeatMorgan2 (FCFP4)': [G.GetMorganGenerator(radius=2, fpSize=2048,
                          atomInvariantsGenerator=G.GetMorganFeatureAtomInvGen()).GetFingerprint(m) for m in mols],
 'AtomPair':         [G.GetAtomPairGenerator(fpSize=2048).GetFingerprint(m) for m in mols],
 'TopoTorsion':      [G.GetTopologicalTorsionGenerator(fpSize=2048).GetFingerprint(m) for m in mols],
 'MACCS':            [MACCSkeys.GenMACCSKeys(m) for m in mols],
 'RDKit path':       [Chem.RDKFingerprint(m, fpSize=2048) for m in mols],
}
metrics = {'Tanimoto': DataStructs.BulkTanimotoSimilarity, 'Dice': DataStructs.BulkDiceSimilarity,
           'Cosine': DataStructs.BulkCosineSimilarity,
           'Tversky(0.9,0.1)': lambda a, bs: DataStructs.BulkTverskySimilarity(a, bs, 0.9, 0.1)}
SIMS = {}
for fn, fl in fps.items():
    for mn, f in metrics.items():
        SIMS[f'2D | {fn} | {mn}'] = np.array([f(fl[i], fl) for i in range(len(fl))])

# ---------- 3D: USR and USRCAT on one MMFF conformer ----------
t = time.time(); usr, usrcat, ok = [], [], 0
for m in mols:
    mh = Chem.AddHs(m)
    p = AllChem.ETKDGv3(); p.randomSeed = 42
    if AllChem.EmbedMolecule(mh, p) != 0:
        p.useRandomCoords = True; AllChem.EmbedMolecule(mh, p)
    try:
        AllChem.MMFFOptimizeMolecule(mh, maxIters=500)
        mh = Chem.RemoveHs(mh)
        usr.append(rdMolDescriptors.GetUSR(mh)); usrcat.append(rdMolDescriptors.GetUSRCAT(mh)); ok += 1
    except Exception:
        usr.append(None); usrcat.append(None)
def usr_matrix(D):
    n = len(D); S = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if D[i] is not None and D[j] is not None:
                S[i, j] = rdMolDescriptors.GetUSRScore(D[i], D[j])
    return S
SIMS['3D | USR | USR score'] = usr_matrix(usr)
SIMS['3D | USRCAT | USR score'] = usr_matrix(usrcat)
print(f'3D descriptors for {ok}/{len(mols)} drugs in {time.time()-t:.0f}s')
np.savez_compressed('sims.npz', **{k.replace(' ', '_').replace('|', 'I'): v for k, v in SIMS.items()})
pd.Series(list(SIMS.keys())).to_csv('sim_names.csv', index=False)
print(len(SIMS), 'similarity measures built for', len(APIS), 'drugs')


## Step 2 — Benchmark: all settings, the honest held-out test, and analysis

In [ ]:
import nbformat, numpy as np, pandas as pd
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')
nb = nbformat.read('IL_Recommender_v2_pKa.ipynb', as_version=4)
code = [c.source for c in nb.cells if c.cell_type == 'code']
g = {}
for src in code[:2]:
    exec('\n'.join(l for l in src.split('\n') if not l.strip().startswith('!')), g)
B, API, POOL, MST = g['B'], g['API'], g['POOL'], g['MST']
names = pd.read_csv('sim_names.csv').iloc[:, 0].tolist()
Z = np.load('sims.npz'); SIMS = {n: Z[n.replace(' ', '_').replace('|', 'I')] for n in names}
APIS = list(B.api_smiles.unique()); aidx = {s: i for i, s in enumerate(APIS)}
pair_api = B.api_smiles.map(aidx).values; pair_ion = B.ion_id.values
chol = MST[MST.smiles == 'C[N+](C)(C)CCO'].master_id.iloc[0]

# evaluation drugs: same 447-pair set as the current recommender
ev = [a for a in APIS if a in API.index and pd.notna(API.loc[a, 'api_pKa']) and API.loc[a, 'role'] in ('acid', 'base')]
pools = {}
for a in ev:
    want = 'cation' if API.loc[a, 'role'] == 'acid' else 'anion'
    ids = POOL[POOL.ion_type == want].master_id.values
    own = B[B.api_smiles == a].ion_id.value_counts()
    pop = B.ion_id.value_counts().reindex(ids).fillna(0).values - own.reindex(ids).fillna(0).values
    pools[a] = (ids, pop, B[B.api_smiles == a].ion_id.values)

def ranks_for(S, k):
    out = {}
    for a in ev:
        i = aidx[a]; ids, pop, truth = pools[a]
        s = S[i, pair_api].astype(float); s[pair_api == i] = -np.inf     # leave this drug out
        top = np.argpartition(-s, k)[:k]
        votes = pd.Series(s[top]).groupby(pair_ion[top]).sum()
        an = votes.reindex(ids).fillna(0).values
        order = np.lexsort((-pop, -an))                                  # analogy first, popularity second
        rank = np.empty(len(ids), int); rank[order] = np.arange(1, len(ids) + 1)
        pos = dict(zip(ids, rank))
        out[a] = [pos.get(t, np.nan) for t in truth]
    return out

def score(R, drugs, hard=False):
    r = [x for a in drugs for x, t in zip(R[a], pools[a][2]) if (not hard or t != chol)]
    r = np.array(r, float)
    return dict(top1=np.mean(r <= 1), top5=np.mean(r <= 5), top10=np.mean(r <= 10), mrr=np.mean(1 / r), n=len(r))

KS = [5, 15, 30]
ALL = {(n, k): ranks_for(SIMS[n], k) for n in names for k in KS}
base = ('2D | Morgan2 (ECFP4) | Tanimoto', 15)
b_all, b_hard = score(ALL[base], ev), score(ALL[base], ev, True)
print(f"CHECK current setting: top-5 {b_all['top5']:.3f} (expect 0.427) | hard top-5 {b_hard['top5']:.3f} (expect 0.305)\n")

rows = []
for (n, k), R in ALL.items():
    a, h = score(R, ev), score(R, ev, True)
    rows.append(dict(measure=n, k=k, top5=a['top5'], mrr=a['mrr'], hard_top5=h['top5'], hard_mrr=h['mrr'], top1=a['top1']))
T = pd.DataFrame(rows).sort_values('hard_mrr', ascending=False)
T.to_csv('similarity_benchmark_full.csv', index=False)
print('Top 12 settings on ALL drugs (sorted by hard-case MRR) -- for reference, NOT the honest score:')
print(T.head(12).round(3).to_string(index=False))

# ---------- honest protocol: choose on 70% of drugs, report on the unseen 30%, 5 splits ----------
print('\n=== Honest test: pick best setting on 70% of drugs, score on the other 30% ===')
res = []
for seed in range(5):
    rng = np.random.default_rng(seed); perm = rng.permutation(ev)
    sel, test = list(perm[:int(.7 * len(ev))]), list(perm[int(.7 * len(ev)):])
    best = max(ALL, key=lambda c: score(ALL[c], sel, True)['mrr'])       # chosen WITHOUT seeing test drugs
    bt, bh = score(ALL[best], test), score(ALL[best], test, True)
    ct, ch = score(ALL[base], test), score(ALL[base], test, True)
    res.append(dict(split=seed, chosen=f'{best[0]} (k={best[1]})', test_top5=bt['top5'], base_top5=ct['top5'],
                    test_hard_top5=bh['top5'], base_hard_top5=ch['top5'], test_hard_mrr=bh['mrr'], base_hard_mrr=ch['mrr']))
Rr = pd.DataFrame(res)
print(Rr.round(3).to_string(index=False))
print('\nMean over 5 splits (unseen drugs):')
for c in ['top5', 'hard_top5', 'hard_mrr']:
    print(f"  {c:10s} chosen {Rr['test_'+c].mean():.3f} +/- {Rr['test_'+c].std():.3f}   vs current {Rr['base_'+c].mean():.3f} +/- {Rr['base_'+c].std():.3f}")

print('\n=== By method family (best k for each, all drugs) ===')
T['fp'] = T.measure.str.split(r' \| ').str[:2].str.join(' | ')
best = T.sort_values('hard_mrr', ascending=False).groupby('fp').head(1).sort_values('hard_mrr', ascending=False)
print(best[['fp', 'measure', 'k', 'top5', 'hard_top5', 'hard_mrr']].round(3).to_string(index=False))

print('\n=== Does the number of neighbours (k) matter? (mean over all 30 measures) ===')
print(T.groupby('k')[['top5', 'hard_top5', 'hard_mrr']].mean().round(3).to_string())

print('\n=== Fusion: average of 2D and 3D similarity ===')
for a2, a3 in [('2D | MACCS | Tanimoto', '3D | USRCAT | USR score'), ('2D | Morgan2 (ECFP4) | Tanimoto', '3D | USRCAT | USR score')]:
    F = (SIMS[a2] + SIMS[a3]) / 2
    R = ranks_for(F, 15); s, h = score(R, ev), score(R, ev, True)
    print(f"{a2.split(' | ')[1]} + USRCAT (k=15): top-5 {s['top5']:.3f}  hard top-5 {h['top5']:.3f}  hard MRR {h['mrr']:.3f}")

print('\n=== WHY: accuracy depends on whether a close neighbour exists (current setting) ===')
S = SIMS['2D | Morgan2 (ECFP4) | Tanimoto']
R = ALL[base]; rows = []
for a in ev:
    i = aidx[a]; s = S[i].copy(); s[i] = -1
    nn = s[[aidx[x] for x in B[B.api_smiles != a].api_smiles.unique()]].max()
    for r, t in zip(R[a], pools[a][2]):
        rows.append(dict(nn=nn, rank=r, hard=(t != chol)))
D = pd.DataFrame(rows); D['closest neighbour similarity'] = pd.cut(D.nn, [0, .3, .5, .7, 1.01], labels=['<0.3', '0.3-0.5', '0.5-0.7', '>0.7'])
print(D.groupby('closest neighbour similarity', observed=True).apply(
    lambda x: pd.Series({'pairs': len(x), 'top-5': np.mean(x['rank'] <= 5), 'hard top-5': np.mean(x[x.hard]['rank'] <= 5)}),
    include_groups=False).round(3).to_string())


## Findings

Exact numbers shift slightly between RDKit versions (fingerprint hashing, 3D conformers), so the check line may
read 0.423–0.427; the conclusions below held in every run.

- **3D shape loses to 2D**: USR and USRCAT are clearly worse than the top 2D fingerprints; shape ignores the
  functional groups that decide counter-ion choice. 2D+3D fusion did not help.
- **Tanimoto vs Dice is identical in effect** — Dice is a monotonic function of Tanimoto, so the same nearest
  drugs are always chosen.
- **MACCS has a small, consistent edge**: chosen in most held-out splits and never meaningfully worse than
  ECFP4 on unseen drugs (about +1 to +3 points hard-case top-5), but within split-to-split variation.
- **The bottleneck is data coverage**: when the closest known drug has similarity < 0.3 (a third of pairs),
  top-5 is ~24%; with a close neighbour it is ~50–58%. Future signals (dipole, H-bonding, pKa rules) should be
  tested as the fallback for these no-neighbour drugs.